In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from google.colab import files

print("=== PRE-PROCESSING & TRAINING DATA DEVELOPMENT ===\n")

# Upload matches.csv to the Colab environment
files.upload()

# Load the dataset and parse datetime
matches = pd.read_csv("matches.csv")
matches["datetime"] = pd.to_datetime(matches["datetime"], errors="coerce")

# Create 'result' column based on home_goals and away_goals
def get_match_result(row):
    if row['home_goals'] > row['away_goals']:
        return 'Home Win'
    elif row['away_goals'] > row['home_goals']:
        return 'Away Win'
    else:
        return 'Draw'

matches['result'] = matches.apply(get_match_result, axis=1)

print("1. INITIAL DATA OVERVIEW")
print("-" * 40)
print(f"Dataset shape: {matches.shape}")
print("\nColumns and data types:")
print(matches.dtypes)
print("\nFirst few rows:")
display(matches.head())

print("\n2. IDENTIFYING FEATURE TYPES")
print("-" * 40)

# Separate features and target variable
X = matches.drop("result", axis=1)
y = matches["result"]

# Identify categorical features
categorical_features = ["home_team", "away_team"]
print("Categorical features:")
print(categorical_features)

# Identify numerical features
numerical_features = ["home_goals", "away_goals", "total_goals", "penalty_goals", "own_goals"]
numerical_features = [c for c in numerical_features if c in X.columns]
print("\nNumerical features:")
print(numerical_features)

# Identify datetime features
datetime_features = ["datetime"]
print("\nDatetime features:")
print(datetime_features)

print("\n3. FEATURE ENGINEERING")
print("-" * 40)

# Extract useful components from datetime
X["year"] = X["datetime"].dt.year
X["month"] = X["datetime"].dt.month
X["day_of_week"] = X["datetime"].dt.dayofweek

# Remove original datetime column after extraction
X = X.drop("datetime", axis=1)

# Update numerical feature list after engineering
numerical_features = numerical_features + ["year", "month", "day_of_week"]

print("New numerical features after engineering:")
print(numerical_features)

print("\n4. HANDLING CATEGORICAL VARIABLES")
print("-" * 40)

# Display cardinality of team name features
print("Original number of unique home teams:", X["home_team"].nunique())
print("Original number of unique away teams:", X["away_team"].nunique())

# Encode team names using label encoding
team_encoder = LabelEncoder()
all_teams = pd.concat([X["home_team"], X["away_team"]]).astype(str).unique()
team_encoder.fit(all_teams)

X["home_team_encoded"] = team_encoder.transform(X["home_team"].astype(str))
X["away_team_encoded"] = team_encoder.transform(X["away_team"].astype(str))

# Remove original categorical team columns
X = X.drop(["home_team", "away_team"], axis=1)

# Add encoded features to numerical feature list
categorical_encoded = ["home_team_encoded", "away_team_encoded"]
numerical_features = numerical_features + categorical_encoded

print("After encoding, categorical features are represented numerically")
print(f"Encoded features: {categorical_encoded}")

print("\n5. STANDARDIZING NUMERICAL FEATURES")
print("-" * 40)

# Display feature ranges before scaling
print("Before scaling - numerical feature ranges:")
for feature in numerical_features:
    if feature in X.columns:
        mn = pd.to_numeric(X[feature], errors="coerce").min()
        mx = pd.to_numeric(X[feature], errors="coerce").max()
        print(f"{feature}: {mn:.2f} to {mx:.2f}")

# Convert all numerical features to numeric type
for feature in numerical_features:
    X[feature] = pd.to_numeric(X[feature], errors="coerce")

# Apply standard scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X[numerical_features])

# Create DataFrame containing scaled features
X_scaled_df = pd.DataFrame(X_scaled, columns=numerical_features, index=X.index)

print("\nAfter scaling, all features have mean approximately 0 and standard deviation approximately 1:")
print(f"Means: {X_scaled_df.mean().values}")
print(f"Standard deviations: {X_scaled_df.std().values}")

print("\n6. SPLITTING INTO TRAINING AND TESTING SETS")
print("-" * 40)

# Encode target labels
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y.astype(str))

print("Target variable mapping:")
for i, class_name in enumerate(target_encoder.classes_):
    print(f"  {i}: {class_name}")

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"Training features: {X_train.shape[1]}")

# Display class distribution in training data
print("\nClass distribution in training set:")
train_class_counts = pd.Series(y_train).value_counts().sort_index()
for i, count in train_class_counts.items():
    class_name = target_encoder.inverse_transform([i])[0]
    print(f"  {class_name}: {count} samples ({count/len(y_train)*100:.1f}%)")

# Display class distribution in testing data
print("\nClass distribution in testing set:")
test_class_counts = pd.Series(y_test).value_counts().sort_index()
for i, count in test_class_counts.items():
    class_name = target_encoder.inverse_transform([i])[0]
    print(f"  {class_name}: {count} samples ({count/len(y_test)*100:.1f}%)")

=== PRE-PROCESSING & TRAINING DATA DEVELOPMENT ===



Saving matches.csv to matches (1).csv
1. INITIAL DATA OVERVIEW
----------------------------------------
Dataset shape: (887, 6)

Columns and data types:
datetime      datetime64[ns]
home_team             object
away_team             object
home_goals             int64
away_goals             int64
result                object
dtype: object

First few rows:


,datetime,home_team,away_team,home_goals,away_goals,result
0,1984-04-08,England,Denmark,2,1,Home Win
1,1984-04-08,Italy,Sweden,2,3,Away Win
2,1984-04-28,Denmark,England,0,1,Away Win
3,1984-04-28,Sweden,Italy,2,1,Home Win
4,1984-05-12,Sweden,England,1,0,Home Win



2. IDENTIFYING FEATURE TYPES
----------------------------------------
Categorical features:
['home_team', 'away_team']

Numerical features:
['home_goals', 'away_goals']

Datetime features:
['datetime']

3. FEATURE ENGINEERING
----------------------------------------
New numerical features after engineering:
['home_goals', 'away_goals', 'year', 'month', 'day_of_week']

4. HANDLING CATEGORICAL VARIABLES
----------------------------------------
Original number of unique home teams: 84
Original number of unique away teams: 84
After encoding, categorical features are represented numerically
Encoded features: ['home_team_encoded', 'away_team_encoded']

5. STANDARDIZING NUMERICAL FEATURES
----------------------------------------
Before scaling - numerical feature ranges:
home_goals: 0.00 to 13.00
away_goals: 0.00 to 10.00
year: 1984.00 to 2025.00
month: 2.00 to 12.00
day_of_week: 0.00 to 6.00
home_team_encoded: 0.00 to 83.00
away_team_encoded: 0.00 to 83.00

After scaling, all features have 